# Patient Health Risk Prediction — Data Analytics & ML
This notebook explores the included **synthetic** dataset, prepares features, trains a Random Forest classifier, and evaluates it. It is an academic demo, not a medical tool.

In [ ]:
import pandas as pd\nimport matplotlib.pyplot as plt\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score\nfrom pathlib import Path\nDATA = Path('../data/synthetic_patient_data.csv')\nif not DATA.exists(): DATA = Path('data/synthetic_patient_data.csv')\ndf = pd.read_csv(DATA)\ndf.head()

## 1. Data overview and quality checks

In [ ]:
print(df.shape)\ndisplay(df.info())\ndisplay(df.isna().sum())\ndisplay(df.describe(include='all').T)

## 2. Exploratory data analysis

In [ ]:
df['risk_level'].value_counts().plot(kind='bar', title='Synthetic risk class distribution')\nplt.xlabel('Risk class'); plt.ylabel('Records'); plt.show()\ndf.hist(figsize=(12,8)); plt.tight_layout(); plt.show()

## 3. Preprocessing and model training

In [ ]:
features=['age','gender','bmi','systolic_bp','diastolic_bp','glucose','cholesterol','heart_rate','smoking','physical_activity','family_history']\nnumeric=['age','bmi','systolic_bp','diastolic_bp','glucose','cholesterol','heart_rate','smoking','family_history']\ncategorical=['gender','physical_activity']\nX=df[features]; y=df['risk_level']\npre=ColumnTransformer([('num',Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]),numeric),('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),categorical)])\npipe=Pipeline([('preprocess',pre),('model',RandomForestClassifier(n_estimators=220,max_depth=12,min_samples_leaf=3,class_weight='balanced',random_state=42))])\nX_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)\npipe.fit(X_train,y_train)\npred=pipe.predict(X_test)\nprint('Accuracy:', accuracy_score(y_test,pred))\nprint(classification_report(y_test,pred))

## 4. Confusion matrix

In [ ]:
ConfusionMatrixDisplay.from_estimator(pipe,X_test,y_test,cmap='Blues')\nplt.title('Random Forest — synthetic test split'); plt.show()

## 5. Example prediction

In [ ]:
sample=X_test.iloc[[0]]\nprint('Predicted class:', pipe.predict(sample)[0])\nprint('Class probabilities:', dict(zip(pipe.classes_, pipe.predict_proba(sample)[0].round(3))))

## Conclusion and limitations
The workflow demonstrates data cleaning, EDA, preprocessing, supervised learning, evaluation, and inference. Because the dataset and labels are synthetic, results do not establish clinical validity. Do not use this model for diagnosis or healthcare decisions.